In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Howard et al. (JCP 493, 2023, 112462)
COMPOSITE MF | LAYER-1 CANONICAL s=0.35 | TABLE C.7 RERUN
==========================================================================
================================================

Purpose
-------
Train ONLY the composite data-driven multifidelity DeepONet used for the
multiresolution ice-sheet experiment in Howard et al., Sec. 3.4.2, on the
independently generated t=50 benchmark dataset.

This script deliberately does NOT run SF20/SF50 and does NOT tune the model
to reproduce Howard et al.'s reported error.  It implements the published
MF method as closely as the paper permits, then evaluates it on OUR fixed
dataset for a fair comparison against our ICL method.

Dataset
-------
Expected Kaggle input root:

    /kaggle/input/notebooks/xli46913/canonical-dataset/LAYER1_CANONICAL_gpScale0p35

Expected master filename somewhere recursively under that folder:

    howard_ice_MF_VERSION_A_t50_gpScale0p35_adaptivePicard.npz

Training budget:
    N_L = 100 low-fidelity functions
    N_H = 20 high-fidelity functions

Test:
    50 held-out HF functions

Published method implemented
----------------------------
Howard et al., Sec. 2 + Sec. 3.4.2 + Appendix C:

1) Composite MF DeepONet with THREE jointly trained blocks:
      * low-fidelity modified DeepONet
      * linear-correlation standard DeepONet, no activations
      * nonlinear-correlation modified DeepONet, activation on every layer

2) Ice operator:
      (beta(x,y), H(x,y)) -> depth-averaged velocity (u(x,y), v(x,y))

3) Grid dimensions:
      M_L = P_L = 2 * 15^2 = 450
      M_H = P_H = 2 * 41^2 = 3362

4) Sec. 3.4.2 / Table C.7 network sizes:
      LF modified DeepONet     : 5 layers, 200 neurons
      linear DeepONet          : 1 layer, 10 neurons
      nonlinear modified net   : 5 layers, 200 neurons

5) Activation:
      tanh unless noted
      LF modified DeepONet: no activation on final layer
      nonlinear modified DeepONet: activation on EVERY layer
      linear DeepONet: NO activations

6) MF optimizer:
      Adam
      JAX exponential decay:
          initial = 1e-4
          decay_steps = 5000
          decay_rate = 0.97

7) MF function batches, Appendix C.3:
      LF batch = 10
      HF batch = 5

8) Data-driven MF loss, Eq. (10):
      L = lambda1 * L_HF
        + lambda2 * L_LF
        + lambda3 * (sum w_nl^2 + sum b_nl^2)
        + lambda4 * (sum w_LF^2 + sum b_LF^2)

   FINAL ADOPTED LOSS WEIGHTS (Table C.7):
          lambda1(HF) = 1
          lambda2(LF) = 10
          lambda3     = 1e-3
          lambda4     = 1e-4

   This file is locked to the Table-C.7 setting so the composite baseline
   and our non-composite adaptation use one consistent reference convention.

9) Composite branch construction, Sec. 2.2:
      Linear branch input:
          learned LF output evaluated at ALL HF output points.

      Nonlinear branch input:
          [HF input function sampled at HF sensors,
           learned LF output evaluated at Q_H points]

      We take Q_H = P_L, as the paper states is typical.

10) Vector velocity output:
      Howard et al. use P_L = 2*15^2 and P_H = 2*41^2.
      For the exact two-component DeepONet contraction, we follow the
      companion ice-sheet DeepONet paper cited by Howard et al.:
          branch/trunk embeddings are in R^(2p)
          first p features -> u
          second p features -> v
      Main latent = 200 -> p=100/component
      Linear latent = 10 -> p=5/component

11) Training length:
      Fig. C.20 spans approximately 0--600,000 iterations.
      Therefore default TRAIN_STEPS = 600_000.

What the paper does NOT uniquely specify
----------------------------------------
The paper does not state its exact preprocessing / nondimensionalization,
parameter initialization, random seed, or authors' source code.

To avoid the serious distortion caused by OUTPUT standardization in earlier
experiments, this script:
    * DOES NOT standardize velocity outputs.
    * Computes Eq. (10) directly in physical velocity units.
    * Only nondimensionalizes NETWORK INPUTS using fixed physical scales:
          beta / 3000
          H / 1000
          coordinates mapped to [-1,1]
      These scales are fixed a priori and are NOT fitted using train/test data.

This input-only nondimensionalization is an implementation choice required
to keep tanh networks numerically usable with meter-scale coordinates and
beta values of O(10^3).  It does not change the physical-unit data losses.

Framework
---------
Howard et al. used JAX.  This script also uses JAX and
jax.example_libraries.optimizers.exponential_decay / adam.

No test set is used for optimization, hyperparameter selection, or checkpoint
selection.  Test evaluation is diagnostic and final only.

References
----------
Howard, Perego, Karniadakis, Stinis,
"Multifidelity deep operator networks for data-driven and physics-informed
problems", Journal of Computational Physics 493 (2023) 112462.

He, Perego, Howard, Karniadakis, Stinis,
"A hybrid deep neural operator/finite element method for ice-sheet modeling",
Journal of Computational Physics 492 (2023) 112428.
"""

from __future__ import annotations

import csv
import json
import hashlib
import math
import pickle
import time
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
from scipy.interpolate import RegularGridInterpolator

import jax
import jax.numpy as jnp
from jax import jit, value_and_grad
from jax.example_libraries import optimizers


# =============================================================================
# CONFIG
# =============================================================================

INPUT_ROOT = "/kaggle/input/notebooks/xli46913/canonical-dataset/LAYER1_CANONICAL_gpScale0p35"
VERSION_A_FILENAME = "howard_ice_MF_VERSION_A_t50_gpScale0p35_adaptivePicard.npz"
NPZ_PATH_FIXED = (
    "/kaggle/input/notebooks/xli46913/canonical-dataset/"
    "LAYER1_CANONICAL_gpScale0p35/"
    "howard_ice_MF_VERSION_A_t50_gpScale0p35_adaptivePicard.npz"
)
EXPECTED_DATASET_SHA256 = "6786500bb1f34939ea3a589cbe5ff1bf23b995eb9a3580cd6cfb11465dfe4875"
SINGLE_VERSION = "A"

SAVE_ROOT = "/kaggle/working/LAYER1_COMPOSITE_MF_CANONICAL_gpScale0p35_TABLE_C7_metadatafix"

# Version A only.

SEED = 20260827

# Fig. C.20 is plotted to ~600k iterations.
TRAIN_STEPS = 600_000

LF_BATCH = 10
HF_BATCH = 5

LOG_EVERY = 1_000
RUN_TEST_DURING_TRAIN = False
CHECKPOINT_EVERY = 100_000
EVAL_BATCH = 5

HIDDEN = 200
LATENT = 200
N_LAYERS = 5
LINEAR_LATENT = 10

LR0 = 1.0e-4
LR_DECAY_STEPS = 5_000
LR_DECAY_RATE = 0.97

# Final unified reference setting: Howard Table C.7.
# Locked here intentionally; do not switch weights inside this final baseline.
LAMBDA_SOURCE = "table_c7"
LAMBDA_HF = 1.0
LAMBDA_LF = 10.0
LAMBDA_NL_REG = 1.0e-3
LAMBDA_LF_REG = 1.0e-4

# Fixed input nondimensionalization only; outputs remain physical m/yr.
BETA_SCALE = 3000.0
H_SCALE = 1000.0

# Paper reference, for information only. Never used for tuning.
HOWARD_PAPER_REFERENCE = {
    "N_L": 100,
    "N_H": 20,
    "test": 50,
    "mean_mse": 0.71,
    "mean_rel_l2": 0.0312,
}


# =============================================================================
# DATA PATH
# =============================================================================

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def resolve_data_path(filename: str) -> Path:
    """
    Canonical Layer-1 comparison: prefer the exact fixed dataset path.
    Refuse to proceed if the file or SHA256 differs from the locked dataset.
    """
    fixed = Path(NPZ_PATH_FIXED)

    if not fixed.is_file():
        raise FileNotFoundError(
            "Canonical Layer-1 dataset not found at the locked path:\n"
            f"  {fixed}"
        )

    if fixed.name != filename:
        raise RuntimeError(
            f"Locked file name mismatch: {fixed.name} != {filename}"
        )

    digest = sha256_file(fixed)
    if digest.lower() != EXPECTED_DATASET_SHA256.lower():
        raise RuntimeError(
            "Canonical Layer-1 dataset SHA256 mismatch.\n"
            f"Expected: {EXPECTED_DATASET_SHA256}\n"
            f"Actual  : {digest}\n"
            f"Path    : {fixed}"
        )

    print(f"[data-path] {filename}\n  -> {fixed}")
    print(f"[dataset SHA256] {digest}")
    return fixed


# =============================================================================
# REPRODUCIBILITY
# =============================================================================

def make_rng(seed: int) -> np.random.Generator:
    return np.random.default_rng(seed)


# =============================================================================
# BASIC DATA HELPERS
# =============================================================================

def flatten_input(beta: np.ndarray, H: np.ndarray) -> np.ndarray:
    """
    [N,ny,nx] beta,H -> [N, 2*ny*nx]
    Fixed input-only nondimensionalization.
    """
    beta_n = np.asarray(beta, dtype=np.float32) / np.float32(BETA_SCALE)
    H_n = np.asarray(H, dtype=np.float32) / np.float32(H_SCALE)
    return np.concatenate(
        [beta_n.reshape(len(beta_n), -1), H_n.reshape(len(H_n), -1)],
        axis=1,
    ).astype(np.float32)


def make_coords(x: np.ndarray, y: np.ndarray) -> np.ndarray:
    """
    Spatial trunk coordinates -> [-1,1]^2.
    This is fixed nondimensionalization, not train/test fitting.
    """
    x = np.asarray(x, dtype=np.float32)
    y = np.asarray(y, dtype=np.float32)
    xmin, xmax = float(x.min()), float(x.max())
    ymin, ymax = float(y.min()), float(y.max())

    xn = 2.0 * (x - xmin) / max(xmax - xmin, 1.0e-12) - 1.0
    yn = 2.0 * (y - ymin) / max(ymax - ymin, 1.0e-12) - 1.0

    X, Y = np.meshgrid(xn, yn, indexing="xy")
    return np.stack([X.ravel(), Y.ravel()], axis=1).astype(np.float32)


def resample_batch(
    fields: np.ndarray,
    x_src: np.ndarray,
    y_src: np.ndarray,
    x_dst: np.ndarray,
    y_dst: np.ndarray,
) -> np.ndarray:
    """
    Bilinear interpolation of [N,ny,nx] regular-grid fields.

    In the composite model, a HF input function u=(beta,H) must also be
    evaluated on LF sensors before entering the learned LF subnet.
    """
    fields = np.asarray(fields, dtype=np.float32)
    x_src = np.asarray(x_src, dtype=np.float64)
    y_src = np.asarray(y_src, dtype=np.float64)
    x_dst = np.asarray(x_dst, dtype=np.float64)
    y_dst = np.asarray(y_dst, dtype=np.float64)

    Xd, Yd = np.meshgrid(x_dst, y_dst, indexing="xy")
    pts = np.column_stack([Yd.ravel(), Xd.ravel()])

    out = np.empty(
        (len(fields), len(y_dst), len(x_dst)),
        dtype=np.float32,
    )

    for i in range(len(fields)):
        f = RegularGridInterpolator(
            (y_src, x_src),
            fields[i].astype(np.float64),
            method="linear",
            bounds_error=True,
        )
        out[i] = f(pts).reshape(len(y_dst), len(x_dst)).astype(np.float32)

    return out


def per_case_rel_l2(pred: np.ndarray, truth: np.ndarray) -> np.ndarray:
    pred = np.asarray(pred, dtype=np.float64)
    truth = np.asarray(truth, dtype=np.float64)
    num = np.linalg.norm((pred - truth).reshape(len(pred), -1), axis=1)
    den = np.linalg.norm(truth.reshape(len(truth), -1), axis=1) + 1.0e-12
    return num / den


def mean_mse(pred: np.ndarray, truth: np.ndarray) -> float:
    """
    Paper Eq. (25): average squared error over all test functions and
    all scalar output values (both velocity components).
    """
    pred = np.asarray(pred, dtype=np.float64)
    truth = np.asarray(truth, dtype=np.float64)
    return float(np.mean((pred - truth) ** 2))


def summarize(pred: np.ndarray, truth: np.ndarray) -> Dict[str, float]:
    r = per_case_rel_l2(pred, truth)
    return {
        "mean_rel_l2": float(r.mean()),
        "median_rel_l2": float(np.median(r)),
        "p90_rel_l2": float(np.quantile(r, 0.90)),
        "max_rel_l2": float(r.max()),
        "mean_mse": mean_mse(pred, truth),
    }


def fmt_stats(s: Dict[str, float]) -> str:
    return (
        f"RelL2 mean={100*s['mean_rel_l2']:.4f}% | "
        f"median={100*s['median_rel_l2']:.4f}% | "
        f"p90={100*s['p90_rel_l2']:.4f}% | "
        f"max={100*s['max_rel_l2']:.4f}% | "
        f"MSE={s['mean_mse']:.6f}"
    )


# =============================================================================
# LOAD MASTER
# =============================================================================

def load_version_a(path: Path) -> Dict[str, np.ndarray]:
    """Version A: one t=50 function per trajectory."""
    z = np.load(path, allow_pickle=False)
    required = [
        "x_lf", "y_lf", "x_hf", "y_hf",
        "beta_lf_train", "H_lf_train", "U_lf_train",
        "beta_hf_train", "H_hf_train", "U_hf_train",
        "beta_hf_test", "H_hf_test", "U_hf_test",
    ]
    missing = [k for k in required if k not in z.files]
    if missing:
        raise KeyError(f"Version A dataset is missing arrays: {missing}")
    d = {k: z[k] for k in z.files}

    assert d["beta_lf_train"].shape == (100, 15, 15)
    assert d["H_lf_train"].shape == (100, 15, 15)
    assert d["U_lf_train"].shape == (100, 2, 15, 15)
    assert d["beta_hf_train"].shape == (20, 41, 41)
    assert d["H_hf_train"].shape == (20, 41, 41)
    assert d["U_hf_train"].shape == (20, 2, 41, 41)
    assert d["beta_hf_test"].shape == (50, 41, 41)
    assert d["H_hf_test"].shape == (50, 41, 41)
    assert d["U_hf_test"].shape == (50, 2, 41, 41)
    return d


def load_version_b(path: Path) -> Dict[str, np.ndarray]:
    """Version B: t=1,...,50 snapshots retained for every trajectory."""
    z = np.load(path, allow_pickle=False)
    required = [
        "x_lf", "y_lf", "x_hf", "y_hf", "times",
        "beta_lf_train", "H_lf_train_traj", "U_lf_train_traj",
        "beta_hf_train", "H_hf_train_traj", "U_hf_train_traj",
        "beta_hf_test", "H_hf_test_t50", "U_hf_test_t50",
    ]
    missing = [k for k in required if k not in z.files]
    if missing:
        raise KeyError(f"Version B dataset is missing arrays: {missing}")
    d = {k: z[k] for k in z.files}

    assert d["times"].shape == (50,)
    assert d["beta_lf_train"].shape == (100, 15, 15)
    assert d["H_lf_train_traj"].shape == (100, 50, 15, 15)
    assert d["U_lf_train_traj"].shape == (100, 50, 2, 15, 15)
    assert d["beta_hf_train"].shape == (20, 41, 41)
    assert d["H_hf_train_traj"].shape == (20, 50, 41, 41)
    assert d["U_hf_train_traj"].shape == (20, 50, 2, 41, 41)
    assert d["beta_hf_test"].shape == (50, 41, 41)
    assert d["H_hf_test_t50"].shape == (50, 41, 41)
    assert d["U_hf_test_t50"].shape == (50, 2, 41, 41)
    return d


def prepare_experiment_data(
    version: str,
    d: Dict[str, np.ndarray],
) -> Dict[str, Any]:
    """
    Convert either dataset interpretation into EXACTLY the same tensor
    interface expected by the shared Howard MF model.

    Nothing below changes model architecture, loss, optimizer, or learning
    rate. The ONLY difference is which training function examples exist.
    """
    x_lf = d["x_lf"].astype(np.float32)
    y_lf = d["y_lf"].astype(np.float32)
    x_hf = d["x_hf"].astype(np.float32)
    y_hf = d["y_hf"].astype(np.float32)

    if version == "A":
        # -------------------------------------------------------------
        # Version A: final-time-only dataset, read DIRECTLY from the
        # s=0.5 controlled-difficulty Version-A NPZ.
        #
        # Training budget:
        #   100 LF functions at t=50
        #    20 HF functions at t=50
        # Test:
        #    50 held-out HF functions at t=50
        # -------------------------------------------------------------
        lf_branch = flatten_input(
            d["beta_lf_train"],
            d["H_lf_train"],
        )
        lf_target = (
            d["U_lf_train"]
            .astype(np.float32)
            .reshape(100, 2, 15 * 15)
        )

        hf_branch = flatten_input(
            d["beta_hf_train"],
            d["H_hf_train"],
        )
        hf_target = (
            d["U_hf_train"]
            .astype(np.float32)
            .reshape(20, 2, 41 * 41)
        )

        hf_test_branch = flatten_input(
            d["beta_hf_test"],
            d["H_hf_test"],
        )
        hf_test_truth = (
            d["U_hf_test"]
            .astype(np.float32)
            .reshape(50, 2, 41 * 41)
        )

        beta_hf_train_for_lf = d["beta_hf_train"]
        H_hf_train_for_lf = d["H_hf_train"]
        beta_hf_test_for_lf = d["beta_hf_test"]
        H_hf_test_for_lf = d["H_hf_test"]

        protocol = "Version A: t=50 only | controlled GP scale s=0.35"
        simulation_budget = "100 LF trajectories + 20 HF trajectories"
        n_lf_functions = 100
        n_hf_functions = 20

    elif version == "B":
        # -------------------------------------------------------------
        # Version B: trajectory-snapshot interpretation.
        # Same simulation budget, but all t=1..50 snapshots are used.
        # 5000 LF function examples + 1000 HF function examples.
        # -------------------------------------------------------------
        n_time = 50

        beta_lf_snap = np.repeat(
            d["beta_lf_train"][:, None, :, :],
            n_time,
            axis=1,
        ).reshape(100 * n_time, 15, 15)
        H_lf_snap = d["H_lf_train_traj"].reshape(
            100 * n_time, 15, 15
        )
        U_lf_snap = d["U_lf_train_traj"].reshape(
            100 * n_time, 2, 15, 15
        )

        lf_branch = flatten_input(beta_lf_snap, H_lf_snap)
        lf_target = U_lf_snap.astype(np.float32).reshape(
            100 * n_time, 2, 15 * 15
        )

        beta_hf_snap = np.repeat(
            d["beta_hf_train"][:, None, :, :],
            n_time,
            axis=1,
        ).reshape(20 * n_time, 41, 41)
        H_hf_snap = d["H_hf_train_traj"].reshape(
            20 * n_time, 41, 41
        )
        U_hf_snap = d["U_hf_train_traj"].reshape(
            20 * n_time, 2, 41, 41
        )

        hf_branch = flatten_input(beta_hf_snap, H_hf_snap)
        hf_target = U_hf_snap.astype(np.float32).reshape(
            20 * n_time, 2, 41 * 41
        )

        # Primary evaluation remains exactly t=50.
        hf_test_branch = flatten_input(
            d["beta_hf_test"],
            d["H_hf_test_t50"],
        )
        hf_test_truth = (
            d["U_hf_test_t50"]
            .astype(np.float32)
            .reshape(50, 2, 41 * 41)
        )

        beta_hf_train_for_lf = beta_hf_snap
        H_hf_train_for_lf = H_hf_snap
        beta_hf_test_for_lf = d["beta_hf_test"]
        H_hf_test_for_lf = d["H_hf_test_t50"]

        protocol = "t=1..50 trajectory snapshots"
        simulation_budget = "100 LF trajectories + 20 HF trajectories"
        n_lf_functions = 5000
        n_hf_functions = 1000

    else:
        raise ValueError("version must be 'A' or 'B'.")

    # -----------------------------------------------------------------
    # SAME Howard composite rule for BOTH versions:
    # evaluate the SAME HF input function on LF sensors.
    # -----------------------------------------------------------------
    beta_hf_train_to_lf = resample_batch(
        beta_hf_train_for_lf,
        x_hf, y_hf, x_lf, y_lf,
    )
    H_hf_train_to_lf = resample_batch(
        H_hf_train_for_lf,
        x_hf, y_hf, x_lf, y_lf,
    )
    beta_hf_test_to_lf = resample_batch(
        beta_hf_test_for_lf,
        x_hf, y_hf, x_lf, y_lf,
    )
    H_hf_test_to_lf = resample_batch(
        H_hf_test_for_lf,
        x_hf, y_hf, x_lf, y_lf,
    )

    hf_lf_sensor = flatten_input(
        beta_hf_train_to_lf,
        H_hf_train_to_lf,
    )
    hf_test_lf_sensor = flatten_input(
        beta_hf_test_to_lf,
        H_hf_test_to_lf,
    )

    coords_lf = make_coords(x_lf, y_lf)
    coords_hf = make_coords(x_hf, y_hf)

    # Common model-interface audit.
    assert lf_branch.shape == (n_lf_functions, 450)
    assert lf_target.shape == (n_lf_functions, 2, 225)
    assert hf_branch.shape == (n_hf_functions, 3362)
    assert hf_target.shape == (n_hf_functions, 2, 1681)
    assert hf_lf_sensor.shape == (n_hf_functions, 450)
    assert hf_test_branch.shape == (50, 3362)
    assert hf_test_lf_sensor.shape == (50, 450)
    assert hf_test_truth.shape == (50, 2, 1681)
    assert coords_lf.shape == (225, 2)
    assert coords_hf.shape == (1681, 2)

    return {
        "lf_branch": lf_branch,
        "lf_target": lf_target,
        "hf_branch": hf_branch,
        "hf_target": hf_target,
        "hf_lf_sensor": hf_lf_sensor,
        "hf_test_branch": hf_test_branch,
        "hf_test_lf_sensor": hf_test_lf_sensor,
        "hf_test_truth": hf_test_truth,
        "coords_lf": coords_lf,
        "coords_hf": coords_hf,
        "n_lf_functions": n_lf_functions,
        "n_hf_functions": n_hf_functions,
        "protocol": protocol,
        "simulation_budget": simulation_budget,
    }


# =============================================================================
# JAX PARAMETER INITIALIZATION
# =============================================================================

glorot_normal = jax.nn.initializers.glorot_normal()
zeros_init = jax.nn.initializers.zeros


def init_dense(key: jax.Array, din: int, dout: int) -> Dict[str, jax.Array]:
    k1, k2 = jax.random.split(key)
    return {
        "W": glorot_normal(k1, (din, dout), jnp.float32),
        "b": zeros_init(k2, (dout,), jnp.float32),
    }


def dense(p: Dict[str, jax.Array], x: jax.Array) -> jax.Array:
    return jnp.matmul(x, p["W"]) + p["b"]


def init_modified_deeponet(
    key: jax.Array,
    branch_dim: int,
    coord_dim: int,
    hidden: int,
    latent: int,
    n_layers: int,
) -> Dict[str, Any]:
    if n_layers < 2:
        raise ValueError("Modified DeepONet needs at least 2 layers.")
    if latent % 2 != 0:
        raise ValueError("Vector-output latent must be even.")

    n_dense = 2 + 2 * n_layers
    keys = iter(jax.random.split(key, n_dense))

    enc_u = init_dense(next(keys), branch_dim, hidden)
    enc_x = init_dense(next(keys), coord_dim, hidden)

    u_layers: List[Dict[str, jax.Array]] = []
    x_layers: List[Dict[str, jax.Array]] = []

    # Layer 1.
    u_layers.append(init_dense(next(keys), branch_dim, hidden))
    x_layers.append(init_dense(next(keys), coord_dim, hidden))

    # Layers 2,...,L-1.
    for _ in range(n_layers - 2):
        u_layers.append(init_dense(next(keys), hidden, hidden))
        x_layers.append(init_dense(next(keys), hidden, hidden))

    # Layer L.
    u_layers.append(init_dense(next(keys), hidden, latent))
    x_layers.append(init_dense(next(keys), hidden, latent))

    return {
        "enc_u": enc_u,
        "enc_x": enc_x,
        "u_layers": tuple(u_layers),
        "x_layers": tuple(x_layers),
    }


def init_linear_deeponet(
    key: jax.Array,
    branch_dim: int,
    coord_dim: int,
    latent: int,
) -> Dict[str, Any]:
    if latent % 2 != 0:
        raise ValueError("Vector-output linear latent must be even.")
    k1, k2 = jax.random.split(key)
    return {
        "branch": init_dense(k1, branch_dim, latent),
        "trunk": init_dense(k2, coord_dim, latent),
    }


def init_composite_model(
    key: jax.Array,
    lf_branch_dim: int,
    hf_branch_dim: int,
    p_lf_scalar: int,
) -> Dict[str, Any]:
    k_lf, k_lin, k_nl = jax.random.split(key, 3)

    return {
        "lf": init_modified_deeponet(
            k_lf,
            branch_dim=lf_branch_dim,
            coord_dim=2,
            hidden=HIDDEN,
            latent=LATENT,
            n_layers=N_LAYERS,
        ),
        "linear": init_linear_deeponet(
            k_lin,
            branch_dim=2 * 41 * 41,
            coord_dim=2,
            latent=LINEAR_LATENT,
        ),
        "nonlinear": init_modified_deeponet(
            k_nl,
            branch_dim=hf_branch_dim + p_lf_scalar,
            coord_dim=2,
            hidden=HIDDEN,
            latent=LATENT,
            n_layers=N_LAYERS,
        ),
    }


# =============================================================================
# PAPER MODIFIED DEEPONET FORWARD
# =============================================================================

def modified_forward(
    params: Dict[str, Any],
    branch: jax.Array,        # [B,M]
    coords: jax.Array,        # [P,2]
    activate_last: bool,
) -> jax.Array:
    """
    Howard Sec. 2.1, Eqs. (2)--(6), with vector-output split.

    Returns [B,2,P].
    """
    U = jnp.tanh(dense(params["enc_u"], branch))     # [B,H]
    V = jnp.tanh(dense(params["enc_x"], coords))     # [P,H]

    Zu = jnp.tanh(dense(params["u_layers"][0], branch))
    Zx = jnp.tanh(dense(params["x_layers"][0], coords))

    Up = U[:, None, :]           # [B,1,H]
    Vp = V[None, :, :]           # [1,P,H]

    Hu = (1.0 - Zu[:, None, :]) * Up + Zu[:, None, :] * Vp
    Hx = (1.0 - Zx[None, :, :]) * Up + Zx[None, :, :] * Vp

    # Layers 2,...,L-1.
    for pu, px in zip(params["u_layers"][1:-1], params["x_layers"][1:-1]):
        Zu = jnp.tanh(dense(pu, Hu))
        Zx = jnp.tanh(dense(px, Hx))
        Hu = (1.0 - Zu) * Up + Zu * Vp
        Hx = (1.0 - Zx) * Up + Zx * Vp

    Bu = dense(params["u_layers"][-1], Hu)
    Tx = dense(params["x_layers"][-1], Hx)

    if activate_last:
        Bu = jnp.tanh(Bu)
        Tx = jnp.tanh(Tx)

    prod = Bu * Tx
    latent = prod.shape[-1]
    half = latent // 2

    u = jnp.sum(prod[..., :half], axis=-1)
    v = jnp.sum(prod[..., half:], axis=-1)

    return jnp.stack([u, v], axis=1)


def linear_forward(
    params: Dict[str, Any],
    branch: jax.Array,
    coords: jax.Array,
) -> jax.Array:
    """
    Standard one-layer DeepONet with no activation.
    Returns [B,2,P].
    """
    b = dense(params["branch"], branch)   # [B,10]
    t = dense(params["trunk"], coords)    # [P,10]

    prod = b[:, None, :] * t[None, :, :]
    half = prod.shape[-1] // 2

    u = jnp.sum(prod[..., :half], axis=-1)
    v = jnp.sum(prod[..., half:], axis=-1)

    return jnp.stack([u, v], axis=1)


def branch_regularization_modified(params: Dict[str, Any]) -> jax.Array:
    """
    Eq. (10): sum of squared weights and biases from the BRANCH network only.

    For a modified DeepONet, we include:
      branch encoder enc_u
      all branch-side u_layers

    We do NOT regularize trunk-side parameters here.
    """
    total = jnp.sum(params["enc_u"]["W"] ** 2) + jnp.sum(params["enc_u"]["b"] ** 2)
    for p in params["u_layers"]:
        total = total + jnp.sum(p["W"] ** 2) + jnp.sum(p["b"] ** 2)
    return total


# =============================================================================
# COMPOSITE MF FORWARD
# =============================================================================

def hf_forward(
    params: Dict[str, Any],
    hf_branch: jax.Array,
    lf_sensor_branch_for_hf: jax.Array,
    coords_lf: jax.Array,
    coords_hf: jax.Array,
) -> Tuple[jax.Array, jax.Array, jax.Array, jax.Array, jax.Array]:
    """
    Composite data-driven MF DeepONet, Howard Sec. 2.2.

    Same HF input function u=(beta,H):
      * evaluated at LF sensors -> learned LF subnet
      * evaluated at HF sensors -> nonlinear branch input

    LF subnet is evaluated:
      * on LF output points for Q_H=P_L -> nonlinear branch augmentation
      * on HF output points -> linear branch input
    """
    lf_at_lf = modified_forward(
        params["lf"],
        lf_sensor_branch_for_hf,
        coords_lf,
        activate_last=False,
    )  # [B,2,225]

    lf_at_hf = modified_forward(
        params["lf"],
        lf_sensor_branch_for_hf,
        coords_hf,
        activate_last=False,
    )  # [B,2,1681]

    lf_lf_flat = lf_at_lf.reshape((lf_at_lf.shape[0], -1))  # [B,450]
    lf_hf_flat = lf_at_hf.reshape((lf_at_hf.shape[0], -1))  # [B,3362]

    lin = linear_forward(
        params["linear"],
        lf_hf_flat,
        coords_hf,
    )

    nl_branch = jnp.concatenate(
        [hf_branch, lf_lf_flat],
        axis=1,
    )

    nl = modified_forward(
        params["nonlinear"],
        nl_branch,
        coords_hf,
        activate_last=True,
    )

    return lin + nl, lin, nl, lf_at_lf, lf_at_hf


# =============================================================================
# LOSS / UPDATE
# =============================================================================

def mf_loss(
    params: Dict[str, Any],
    lf_branch_batch: jax.Array,
    lf_target_batch: jax.Array,
    hf_branch_batch: jax.Array,
    hf_lf_sensor_batch: jax.Array,
    hf_target_batch: jax.Array,
    coords_lf: jax.Array,
    coords_hf: jax.Array,
) -> Tuple[jax.Array, Dict[str, jax.Array]]:
    """
    Howard Eq. (10), physical velocity units.
    """
    pred_lf = modified_forward(
        params["lf"],
        lf_branch_batch,
        coords_lf,
        activate_last=False,
    )

    pred_hf, lin, nl, _, _ = hf_forward(
        params,
        hf_branch_batch,
        hf_lf_sensor_batch,
        coords_lf,
        coords_hf,
    )

    # Eq. (8) / Eq. (9): scalar MSE across functions and scalar outputs.
    loss_lf = jnp.mean((pred_lf - lf_target_batch) ** 2)
    loss_hf = jnp.mean((pred_hf - hf_target_batch) ** 2)

    reg_nl = branch_regularization_modified(params["nonlinear"])
    reg_lf = branch_regularization_modified(params["lf"])

    total = (
        LAMBDA_HF * loss_hf
        + LAMBDA_LF * loss_lf
        + LAMBDA_NL_REG * reg_nl
        + LAMBDA_LF_REG * reg_lf
    )

    aux = {
        "loss_hf": loss_hf,
        "loss_lf": loss_lf,
        "reg_nl": reg_nl,
        "reg_lf": reg_lf,
        "lin_rms": jnp.sqrt(jnp.mean(lin ** 2)),
        "nl_rms": jnp.sqrt(jnp.mean(nl ** 2)),
    }
    return total, aux


# JAX optimizer EXACT style used in paper description.
lr_schedule = optimizers.exponential_decay(
    LR0,
    LR_DECAY_STEPS,
    LR_DECAY_RATE,
)

opt_init, opt_update, get_params = optimizers.adam(lr_schedule)


@jit
def update_step(
    step: jax.Array,
    opt_state: Any,
    lf_branch_batch: jax.Array,
    lf_target_batch: jax.Array,
    hf_branch_batch: jax.Array,
    hf_lf_sensor_batch: jax.Array,
    hf_target_batch: jax.Array,
    coords_lf: jax.Array,
    coords_hf: jax.Array,
) -> Tuple[Any, jax.Array, Dict[str, jax.Array]]:
    params = get_params(opt_state)

    (loss, aux), grads = value_and_grad(
        mf_loss,
        has_aux=True,
    )(
        params,
        lf_branch_batch,
        lf_target_batch,
        hf_branch_batch,
        hf_lf_sensor_batch,
        hf_target_batch,
        coords_lf,
        coords_hf,
    )

    opt_state = opt_update(step, grads, opt_state)
    return opt_state, loss, aux


# =============================================================================
# EVALUATION
# =============================================================================

@jit
def predict_hf_jit(
    params: Dict[str, Any],
    hf_branch: jax.Array,
    hf_lf_sensor: jax.Array,
    coords_lf: jax.Array,
    coords_hf: jax.Array,
) -> jax.Array:
    pred, _, _, _, _ = hf_forward(
        params,
        hf_branch,
        hf_lf_sensor,
        coords_lf,
        coords_hf,
    )
    return pred


@jit
def predict_lf_jit(
    params: Dict[str, Any],
    lf_branch: jax.Array,
    coords_lf: jax.Array,
) -> jax.Array:
    return modified_forward(
        params["lf"],
        lf_branch,
        coords_lf,
        activate_last=False,
    )


def batched_hf_predict(
    params: Dict[str, Any],
    hf_branch: np.ndarray,
    hf_lf_sensor: np.ndarray,
    coords_lf: jax.Array,
    coords_hf: jax.Array,
) -> np.ndarray:
    out = []
    for s in range(0, len(hf_branch), EVAL_BATCH):
        e = min(s + EVAL_BATCH, len(hf_branch))
        p = predict_hf_jit(
            params,
            jnp.asarray(hf_branch[s:e]),
            jnp.asarray(hf_lf_sensor[s:e]),
            coords_lf,
            coords_hf,
        )
        out.append(np.asarray(jax.device_get(p), dtype=np.float32))
    return np.concatenate(out, axis=0)


# =============================================================================
# CHECKPOINT / HISTORY
# =============================================================================

def tree_to_numpy(tree: Any) -> Any:
    return jax.tree_util.tree_map(
        lambda x: np.asarray(jax.device_get(x)),
        tree,
    )


def save_checkpoint(
    path: Path,
    params: Dict[str, Any],
    step: int,
    config: Dict[str, Any],
) -> None:
    payload = {
        "step": int(step),
        "params": tree_to_numpy(params),
        "config": config,
    }
    with open(path, "wb") as f:
        pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)


def save_history(path: Path, history: List[Dict[str, float]]) -> None:
    if not history:
        return
    with open(path, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=list(history[0].keys()),
        )
        writer.writeheader()
        writer.writerows(history)


# =============================================================================
# MAIN
# =============================================================================

def run_experiment(
    version: str,
    data_path: Path,
    d: Dict[str, np.ndarray],
) -> Dict[str, Any]:
    """
    Run one dataset interpretation with the EXACT SAME shared model code.

    The model initialization seed, architecture, loss, optimizer, batches,
    learning-rate schedule, training steps, and evaluation code are identical
    across Versions A and B.
    """
    tag = "version_A_t50_only" if version == "A" else "version_B_trajectory"
    save_root = Path(SAVE_ROOT) / tag
    save_root.mkdir(parents=True, exist_ok=True)

    data = prepare_experiment_data(version, d)

    lf_branch_np = data["lf_branch"]
    lf_target_np = data["lf_target"]
    hf_branch_np = data["hf_branch"]
    hf_target_np = data["hf_target"]
    hf_lf_sensor_np = data["hf_lf_sensor"]
    hf_test_branch_np = data["hf_test_branch"]
    hf_test_lf_sensor_np = data["hf_test_lf_sensor"]
    hf_test_truth_np = data["hf_test_truth"]

    coords_lf = jnp.asarray(data["coords_lf"])
    coords_hf = jnp.asarray(data["coords_hf"])

    print("\n" + "=" * 116)
    print(f"HOWARD MF | VERSION {version} | SAME MODEL / SAME SEED / SAME OPTIMIZATION")
    print("=" * 116)
    print("Dataset            :", data_path)
    print("Protocol           :", data["protocol"])
    print("Simulation budget  :", data["simulation_budget"])
    print("LF train functions :", data["n_lf_functions"])
    print("HF train functions :", data["n_hf_functions"])
    print("Primary test       : same 50 held-out HF trajectories at t=50")
    print("Model              : LF modified 5x200 + linear 1x10 + nonlinear modified 5x200")
    print("Model seed         :", SEED, "(RESET identically for each version)")
    print("Batches            : LF=10, HF=5")
    print("Optimizer           : JAX Adam")
    print("LR schedule         : 1e-4, decay_steps=5000, decay_rate=0.97")
    print("Training steps      :", TRAIN_STEPS)
    print("Lambda source       :", LAMBDA_SOURCE)
    print(
        "Loss weights       : "
        f"HF={LAMBDA_HF:g}, LF={LAMBDA_LF:g}, "
        f"NLreg={LAMBDA_NL_REG:g}, LFreg={LAMBDA_LF_REG:g}"
    )
    print("Output scaling      : NONE; physical m/yr")
    print("Input nondim        : beta/3000, H/1000, coords -> [-1,1]")
    print("Checkpoint select   : final step only")
    print("Test during train   :", RUN_TEST_DURING_TRAIN)
    print("=" * 116)

    print("\n[tensor audit]")
    print("  LF branch/target :", lf_branch_np.shape, lf_target_np.shape)
    print("  HF branch/target :", hf_branch_np.shape, hf_target_np.shape)
    print("  HF->LF sensors   :", hf_lf_sensor_np.shape)
    print("  Test tensors     :", hf_test_branch_np.shape, hf_test_truth_np.shape)

    # EXACT SAME initialization for A and B.
    key = jax.random.PRNGKey(SEED)
    params0 = init_composite_model(
        key,
        lf_branch_dim=450,
        hf_branch_dim=3362,
        p_lf_scalar=450,
    )
    opt_state = opt_init(params0)

    leaves = jax.tree_util.tree_leaves(params0)
    n_params = int(sum(np.prod(x.shape) for x in leaves))
    print("\n[shared model audit]")
    print("  Parameters                    :", f"{n_params:,}")
    print("  LF branch input               : 450")
    print("  Nonlinear augmented input     : 3812 = 3362 + 450")
    print("  Linear branch input           : 3362")
    print("  Q_H                           : 450")
    print("  Main latent                   : 200 = 100(u)+100(v)")
    print("  Linear latent                 : 10 = 5(u)+5(v)")
    print("  Global output bias            : NONE")

    config = {
        "version": version,
        "protocol": data["protocol"],
        "dataset": str(data_path),
        "dataset_sha256": sha256_file(data_path),
        "expected_dataset_sha256": EXPECTED_DATASET_SHA256,
        "simulation_budget": data["simulation_budget"],
        "N_L_training_functions": data["n_lf_functions"],
        "N_H_training_functions": data["n_hf_functions"],
        "N_test_t50": 50,
        "paper": "Howard et al., JCP 493 (2023) 112462",
        "section": "Sec. 3.4.2",
        "M_L": 450,
        "P_L": 450,
        "M_H": 3362,
        "P_H": 3362,
        "Q_H": 450,
        "lf_layers": 5,
        "lf_hidden": 200,
        "main_latent": 200,
        "linear_layers": 1,
        "linear_latent": 10,
        "nonlinear_layers": 5,
        "nonlinear_hidden": 200,
        "activation": "tanh",
        "nonlinear_activate_last": True,
        "linear_activation": "none",
        "optimizer": "JAX adam",
        "seed": SEED,
        "lr0": LR0,
        "decay_steps": LR_DECAY_STEPS,
        "decay_rate": LR_DECAY_RATE,
        "steps": TRAIN_STEPS,
        "lf_batch": LF_BATCH,
        "hf_batch": HF_BATCH,
        "lambda_source": LAMBDA_SOURCE,
        "lambda_hf": LAMBDA_HF,
        "lambda_lf": LAMBDA_LF,
        "lambda_nl_reg": LAMBDA_NL_REG,
        "lambda_lf_reg": LAMBDA_LF_REG,
        "regularization_scope": (
            "nonlinear branch-side encoder+branch layers only; "
            "LF branch-side encoder+branch layers only"
        ),
        "vector_output": (
            "first half latent=u, second half=v"
        ),
        "hf_composite_input_rule": (
            "same HF (beta,H) function interpolated to LF sensors "
            "for learned LF subnet"
        ),
        "output_scaling": "none; physical m/yr",
        "input_nondim": {
            "beta_divisor": BETA_SCALE,
            "H_divisor": H_SCALE,
            "coords": "mapped to [-1,1]",
        },
        "checkpoint_selection": "final step only; held-out test evaluated only after training",
        "run_test_during_train": RUN_TEST_DURING_TRAIN,
    }
    with open(save_root / "config.json", "w") as f:
        json.dump(config, f, indent=2)

    # SAME RNG seed reset for each version.
    rng = make_rng(SEED)
    history: List[Dict[str, float]] = []
    t0 = time.time()
    last_loss = math.nan
    last_aux: Dict[str, float] = {}

    n_lf_pool = data["n_lf_functions"]
    n_hf_pool = data["n_hf_functions"]

    print("\n[train] Starting...")

    for step in range(TRAIN_STEPS):
        il = rng.choice(n_lf_pool, size=LF_BATCH, replace=False)
        ih = rng.choice(n_hf_pool, size=HF_BATCH, replace=False)

        opt_state, loss, aux = update_step(
            jnp.asarray(step, dtype=jnp.int32),
            opt_state,
            jnp.asarray(lf_branch_np[il]),
            jnp.asarray(lf_target_np[il]),
            jnp.asarray(hf_branch_np[ih]),
            jnp.asarray(hf_lf_sensor_np[ih]),
            jnp.asarray(hf_target_np[ih]),
            coords_lf,
            coords_hf,
        )

        human_step = step + 1

        if (
            human_step == 1
            or human_step % LOG_EVERY == 0
            or human_step == TRAIN_STEPS
        ):
            loss_h = float(jax.device_get(aux["loss_hf"]))
            loss_l = float(jax.device_get(aux["loss_lf"]))
            reg_nl = float(jax.device_get(aux["reg_nl"]))
            reg_lf = float(jax.device_get(aux["reg_lf"]))
            lin_rms = float(jax.device_get(aux["lin_rms"]))
            nl_rms = float(jax.device_get(aux["nl_rms"]))
            last_loss = float(jax.device_get(loss))

            print(
                f"step {human_step:06d}/{TRAIN_STEPS} | "
                f"total={last_loss:.6e} | "
                f"HF={loss_h:.6e} | LF={loss_l:.6e} | "
                f"regNL={reg_nl:.3e} | regLF={reg_lf:.3e} | "
                f"wHF={LAMBDA_HF*loss_h:.3e} | "
                f"wLF={LAMBDA_LF*loss_l:.3e} | "
                f"wNLreg={LAMBDA_NL_REG*reg_nl:.3e} | "
                f"wLFreg={LAMBDA_LF_REG*reg_lf:.3e} | "
                f"linRMS={lin_rms:.3e} | nlRMS={nl_rms:.3e} | "
                f"lr={float(lr_schedule(step)):.3e} | "
                f"elapsed={(time.time()-t0)/60.0:.1f}m"
            )

            last_aux = {
                "loss_hf": loss_h,
                "loss_lf": loss_l,
                "reg_nl": reg_nl,
                "reg_lf": reg_lf,
                "lin_rms": lin_rms,
                "nl_rms": nl_rms,
            }

        if RUN_TEST_DURING_TRAIN and human_step % 50_000 == 0:
            params_now = get_params(opt_state)
            pred_test = batched_hf_predict(
                params_now,
                hf_test_branch_np,
                hf_test_lf_sensor_np,
                coords_lf,
                coords_hf,
            )
            st = summarize(pred_test, hf_test_truth_np)
            print("  TEST DIAGNOSTIC:", fmt_stats(st))

            history.append({
                "step": human_step,
                "total_loss_last_batch": float(last_loss),
                "loss_hf_last_batch": float(last_aux.get("loss_hf", np.nan)),
                "loss_lf_last_batch": float(last_aux.get("loss_lf", np.nan)),
                "reg_nl_last_batch": float(last_aux.get("reg_nl", np.nan)),
                "reg_lf_last_batch": float(last_aux.get("reg_lf", np.nan)),
                "lin_rms_last_batch": float(last_aux.get("lin_rms", np.nan)),
                "nl_rms_last_batch": float(last_aux.get("nl_rms", np.nan)),
                "lr": float(lr_schedule(step)),
                **st,
                "elapsed_min": (time.time() - t0) / 60.0,
            })
            save_history(save_root / "history.csv", history)

        if (
            human_step % CHECKPOINT_EVERY == 0
            or human_step == TRAIN_STEPS
        ):
            save_checkpoint(
                save_root / f"step{human_step:06d}.pkl",
                get_params(opt_state),
                human_step,
                config,
            )
            print(f"  [checkpoint] saved step {human_step}")

    params_final = get_params(opt_state)
    pred_test = batched_hf_predict(
        params_final,
        hf_test_branch_np,
        hf_test_lf_sensor_np,
        coords_lf,
        coords_hf,
    )
    final = summarize(pred_test, hf_test_truth_np)

    print("\n" + "=" * 116)
    print("OFFICIAL FINAL COMPOSITE MF | LAYER-1 CANONICAL s=0.35 | TABLE C.7")
    print(fmt_stats(final))
    print("=" * 116)

    save_checkpoint(
        save_root / "final.pkl",
        params_final,
        TRAIN_STEPS,
        config,
    )

    rel = per_case_rel_l2(pred_test, hf_test_truth_np)
    np.savez_compressed(
        save_root / "test_predictions.npz",
        prediction=pred_test.astype(np.float32),
        target=hf_test_truth_np.astype(np.float32),
        rel_l2=rel.astype(np.float64),
    )

    summary = {
        "version": version,
        "final_test": final,
        "elapsed_min": (time.time() - t0) / 60.0,
        "config": config,
    }
    with open(save_root / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    return summary


def main() -> None:
    print("=" * 116)
    print("HOWARD COMPOSITE MF | LAYER-1 CANONICAL s=0.35 | TABLE C.7 RERUN")
    print("=" * 116)
    print("JAX devices :", jax.devices())
    print("Input root  :", INPUT_ROOT)
    print("Dataset file:", VERSION_A_FILENAME)
    print("Fixed path  :", NPZ_PATH_FIXED)
    print("Expected SHA:", EXPECTED_DATASET_SHA256)
    print("Protocol    : t=50 only")
    print()
    print("CONTROLLED COMPARISON:")
    print("  Dataset difficulty change relative to the harder benchmark:")
    print("      log-beta GP perturbation scale s = 0.35")
    print("  Howard MF method itself is UNCHANGED:")
    print("      same architecture")
    print("      same initialization seed")
    print("      same Eq.(10) loss")
    print("      same optimizer / LR schedule")
    print("      same LF/HF batches")
    print("      same 600,000 training steps")
    print("      held-out test evaluated only after training")
    print("=" * 116)

    data_path = resolve_data_path(VERSION_A_FILENAME)
    d = load_version_a(data_path)

    # Canonical scientific metadata audit.
    #
    # The canonical Layer-1 generator stores metadata under "metadata_json"
    # (not "config_json").  This is the same metadata field used by the
    # canonical Raw-LF and ICL scripts.
    if "metadata_json" not in d:
        raise RuntimeError(
            "Canonical dataset is missing metadata_json."
        )

    raw = d["metadata_json"]
    if isinstance(raw, np.ndarray):
        raw = raw.item()
    meta = json.loads(str(raw))

    gp_scale = meta.get(
        "beta_gp_scale",
        meta.get("config", {}).get("beta_gp_scale"),
    )
    if gp_scale is None:
        raise RuntimeError(
            "metadata_json does not contain beta_gp_scale."
        )

    gp_scale = float(gp_scale)
    if not np.isfinite(gp_scale) or abs(gp_scale - 0.35) > 1.0e-12:
        raise RuntimeError(
            f"Expected beta_gp_scale=0.35, found {gp_scale}"
        )

    version = str(meta.get("version", ""))
    if version and version != "A_t50_only":
        raise RuntimeError(
            f"Expected Version-A t=50 dataset metadata, found version={version!r}"
        )

    print("[dataset audit] metadata key  = metadata_json")
    print("[dataset audit] version       =", version or "(not explicitly stored)")
    print("[dataset audit] beta_gp_scale =", gp_scale)
    print("[dataset audit] SHA256        =", sha256_file(data_path))

    summary = run_experiment("A", data_path, d)

    root = Path(SAVE_ROOT)
    root.mkdir(parents=True, exist_ok=True)
    with open(root / "single_run_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("\nDONE")


if __name__ == "__main__":
    main()


HOWARD COMPOSITE MF | LAYER-1 CANONICAL s=0.35 | TABLE C.7 RERUN
JAX devices : [CudaDevice(id=0), CudaDevice(id=1)]
Input root  : /kaggle/input/notebooks/xli46913/canonical-dataset/LAYER1_CANONICAL_gpScale0p35
Dataset file: howard_ice_MF_VERSION_A_t50_gpScale0p35_adaptivePicard.npz
Fixed path  : /kaggle/input/notebooks/xli46913/canonical-dataset/LAYER1_CANONICAL_gpScale0p35/howard_ice_MF_VERSION_A_t50_gpScale0p35_adaptivePicard.npz
Expected SHA: 6786500bb1f34939ea3a589cbe5ff1bf23b995eb9a3580cd6cfb11465dfe4875
Protocol    : t=50 only

CONTROLLED COMPARISON:
  Dataset difficulty change relative to the harder benchmark:
      log-beta GP perturbation scale s = 0.35
  Howard MF method itself is UNCHANGED:
      same architecture
      same initialization seed
      same Eq.(10) loss
      same optimizer / LR schedule
      same LF/HF batches
      same 600,000 training steps
      held-out test evaluated only after training
[data-path] howard_ice_MF_VERSION_A_t50_gpScale0p35_adaptivePicard